In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from bertopic import BERTopic
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set PyTorch seed if available
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(SEED)
    # Set deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    pass

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print(f"✅ Random seed set to {SEED} for reproducibility")
print("All random number generators have been seeded.")

✅ Random seed set to 42 for reproducibility
All random number generators have been seeded.


In [2]:
# Read the survey data
df = pd.read_csv('test_survey.csv')

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()

Dataset shape: (203, 1)

Column names:
['Q7_other: How much does each of the following factors attract you to go fishing at a particular site?']

First few rows:


,Q7_other: How much does each of the following factors attract you to go fishing at a particular site?
0,10 mph lake speed limits
1,ability for family fishing
2,ability to catch and KEEP fish species present.
3,Access Boat ramps available
4,access to fish


In [3]:
# Load and prepare the data
df = pd.read_csv('test_survey.csv')

# The first column contains the question, and the second column contains responses
# Extract the column with responses (assuming it's the second column)
responses = df.iloc[:, 0].dropna().astype(str).tolist()

# Remove the question header (first row)
responses = responses[1:]

print(f"Total responses: {len(responses)}")
print(f"\nSample responses:")
for i, resp in enumerate(responses[:5]):
    print(f"{i+1}. {resp}")

Total responses: 202

Sample responses:
1. ability for family fishing
2. ability to catch and KEEP fish species present.
3. Access Boat ramps available
4. access to fish
5. affordability


In [4]:
# Train BERTopic model with hierarchical topics enabled
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# Set seed for embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create UMAP model with seed for reproducibility
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=SEED
)

# Create HDBSCAN model with deterministic parameters
hdbscan_model = HDBSCAN(
    min_cluster_size=3,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# Create vectorizer
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))

# Create BERTopic model with all seeded components
topic_model = BERTopic(
    language="english",
    calculate_probabilities=True,
    verbose=True,
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=None  # Don't reduce topics automatically for reproducibility
)

# Fit the model
topics, probs = topic_model.fit_transform(responses)

print(f"\nNumber of topics found: {len(set(topics)) - 1}")  # -1 to exclude outlier topic
print(f"Number of outliers (topic -1): {sum(1 for t in topics if t == -1)}")
print(f"\nTopic distribution:")
print(pd.Series(topics).value_counts().sort_index())

2025-10-24 21:39:38,167 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

2025-10-24 21:39:39,295 - BERTopic - Embedding - Completed ✓
2025-10-24 21:39:39,296 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2025-10-24 21:39:43,294 - BERTopic - Dimensionality - Completed ✓
2025-10-24 21:39:43,294 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-24 21:39:43,319 - BERTopic - Cluster - Completed ✓
2025-10-24 21:39:43,321 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-24 21:39:43,340 - BERTopic - Representation - Completed ✓



Number of topics found: 20
Number of outliers (topic -1): 31

Topic distribution:
-1     31
 0     19
 1     19
 2     18
 3     11
 4     11
 5     10
 6      9
 7      8
 8      8
 9      7
 10     7
 11     6
 12     6
 13     5
 14     5
 15     5
 16     5
 17     5
 18     4
 19     3
Name: count, dtype: int64


In [5]:
# View initial topics and their top words
topic_info = topic_model.get_topic_info()

print("Initial Topics Found:")
for topic_id in sorted(topic_info['Topic'].unique()):
    if topic_id != -1:  # Skip outliers for now
        print(f"\nTopic {topic_id}:")
        words = topic_model.get_topic(topic_id)
        if words:
            print(f"  Top words: {', '.join([word for word, score in words[:10]])}")
            print(f"  Document count: {len([t for t in topics if t == topic_id])}")

print(f"\nTotal topics: {len(set(topics)) - 1}")
print(f"Outliers: {sum(1 for t in topics if t == -1)}")

Initial Topics Found:

Topic 0:
  Top words: boat, boats, ramps, launch, boat ramps, boat launch, motorized, boat ramp, good boat, motor
  Document count: 19

Topic 1:
  Top words: fish, rainbow, trout, fish fish, biting, salmon, fish biting, caught, fishing, love
  Document count: 19

Topic 2:
  Top words: access, ease, public access, public, access ease, ease access, property, private property, private, free
  Document count: 18

Topic 3:
  Top words: food, fish, able, able fish, barbless, allowed, catch fish, catch, fish food, hooks catch
  Document count: 11

Topic 4:
  Top words: regulations, guides, service, day, avoiding, figure, posted, figure regulations, port, unlimited regulations
  Document count: 11

Topic 5:
  Top words: size, quality fish, quality, fish, size fish, fish size, fish quality, trophy, fish health, health
  Document count: 10

Topic 6:
  Top words: catch, catch release, release, fly, friendly large, group available, group, thing, thing grand, artificial
  Doc

In [6]:
# Set up OpenAI API key for GPT-4o mini
import os
import getpass

# Prompt for API key directly
print("OpenAI API Key Setup")
print("=" * 50)

if 'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY']:
    print("✅ API key already set!")
    print(f"Key starts with: {os.environ['OPENAI_API_KEY'][:10]}...")
    
    reset = input("\nDo you want to enter a different key? (y/n): ").strip().lower()
    if reset != 'y':
        print("Using existing key.")
    else:
        api_key = getpass.getpass("Enter your OpenAI API key: ")
        os.environ['OPENAI_API_KEY'] = api_key
        print("✅ API key updated!")
else:
    print("Please enter your OpenAI API key.")
    print("(Get one at: https://platform.openai.com/api-keys)")
    
    api_key = getpass.getpass("\nEnter your OpenAI API key: ")
    os.environ['OPENAI_API_KEY'] = api_key
    print("✅ API key set successfully!")
    print(f"Key starts with: {api_key[:10]}...")

OpenAI API Key Setup
Please enter your OpenAI API key.
(Get one at: https://platform.openai.com/api-keys)
✅ API key set successfully!
Key starts with: sk-proj-By...


In [7]:
# Generate concise, meaningful topic labels using an LLM with GPU support
import os
import torch

# Check for GPU/MPS availability (Apple Silicon M3 Pro)
if torch.backends.mps.is_available():
    device = "mps"  # Apple Metal Performance Shaders for M3 Pro
    print(f"✅ Using Apple Silicon GPU (Metal) for acceleration")
elif torch.cuda.is_available():
    device = "cuda"
    print(f"✅ Using NVIDIA CUDA GPU")
else:
    device = "cpu"
    print(f"⚠️ Using CPU (no GPU detected)")

# Check if OpenAI API key is available
if 'OPENAI_API_KEY' in os.environ and os.environ['OPENAI_API_KEY']:
    print("✅ OpenAI API key detected - using GPT-4o mini")
    
    from openai import OpenAI as OpenAIClient
    from bertopic.representation import OpenAI
    
    # Create OpenAI client
    client = OpenAIClient(api_key=os.environ['OPENAI_API_KEY'])
    
    # Create representation model with the client
    # Note: OpenAI API calls may have slight variations even with seed
    # For full reproducibility, consider saving and loading the topic model
    representation_model = OpenAI(
        client=client,
        model="gpt-4o-mini",
        chat=True,
        nr_docs=10,  # Use more documents to better understand the topic
        diversity=0.5,  # Encourage diverse keyword selection
        delay_in_seconds=2,  # Rate limiting to avoid API errors
        exponential_backoff=True,
        prompt="""
I have a topic that is part of a survey about fishing site preferences. 

The topic contains these keywords: [KEYWORDS]

Representative survey responses from this topic:
[DOCUMENTS]

Your task: Create a concise, distinctive label that captures what makes THIS specific topic unique and different from other fishing-related topics.

Important guidelines:
- DO NOT use generic labels like "Fishing Catch" or "Fishing Preferences" - be SPECIFIC to what distinguishes this topic
- Focus on the PRIMARY concern or theme that unites these responses
- Use natural, human-readable language that someone would actually say
- Be concise but descriptive enough to differentiate this topic from others
- Look at the actual survey responses to understand the nuance

Good examples of DISTINCT labels:
- If responses focus on launching boats: "Boat Launch Facilities" (not "Fishing")
- If responses focus on keeping fish to eat: "Harvest for Food" (not "Fishing Catch")
- If responses focus on time/season: "Seasonal Timing" (not "Fishing")
- If responses focus on proximity to home: "Location Convenience" (not "Fishing")
- If responses focus on disabilities: "Accessibility Accommodations" (not "Fishing Access")
- If responses focus on scenery/nature: "Outdoor Experience" (not "Fishing")
- If responses focus on regulations: "Regulatory Concerns" (not "Fishing Rules")

Generate ONLY the label - nothing else:"""
    )
    
    print("⚠️ Note: OpenAI API responses may vary slightly even with the same inputs.")
    print("   For 100% reproducibility, save the model after generating labels.")
    
else:
    print("⚠️ No OpenAI API key found - using local Flan-T5-large model")
    print("(No API key required - completely free)")
    
    from transformers import pipeline, set_seed
    from bertopic.representation import TextGeneration
    
    # Set seed for transformers
    set_seed(SEED)
    
    # Map device for transformers pipeline
    if device == "mps":
        pipeline_device = 0  # Use MPS backend
        os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    elif device == "cuda":
        pipeline_device = 0  # Use CUDA GPU
    else:
        pipeline_device = -1  # CPU
    
    generator = pipeline(
        'text2text-generation', 
        model='google/flan-t5-large',
        max_length=50,
        device=pipeline_device
    )
    
    prompt = """
Topic keywords: [KEYWORDS]
Example responses: [DOCUMENTS]

Create a SPECIFIC, DISTINCTIVE label for this fishing survey topic.
Focus on what makes THIS topic unique - not just "fishing" or "catch".
Look at the actual responses to understand the specific concern.

Be concise but capture the human meaning.

Examples of good distinct labels:
- "Boat Launch Facilities" (if about ramps/access)
- "Harvest for Food" (if about keeping fish to eat)
- "Seasonal Timing" (if about weather/seasons)
- "Location Convenience" (if about proximity)

Label:"""
    
    representation_model = TextGeneration(generator, prompt=prompt, nr_docs=10, diversity=0.5)
    print("✅ Local model is fully deterministic with the seed set.")

# Update topics with LLM-generated labels
print("\n🔄 Generating distinctive topic labels with LLM... This may take a minute.")
topic_model.update_topics(responses, representation_model=representation_model)

# Get updated topic info
topic_info = topic_model.get_topic_info()

print("\n✅ Topic labels generated successfully!")
print("\nTopics with LLM-Generated Distinctive Labels:")
print(topic_info[['Topic', 'Count', 'Name']].to_string())

# Show detailed view of top topics
print("\n" + "="*80)
print("TOP TOPICS WITH DISTINCTIVE LABELS")
print("="*80)
for _, row in topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False).head(10).iterrows():
    topic_id = row['Topic']
    print(f"\n📌 {row['Name']} ({row['Count']} responses)")
    print(f"   Keywords: {', '.join([word for word, _ in topic_model.get_topic(topic_id)[:8]])}")
    topic_docs = [responses[i] for i, t in enumerate(topics) if t == topic_id][:3]
    print(f"   Examples:")
    for doc in topic_docs:
        print(f"     • {doc[:90]}...")
    print("-"*80)

✅ Using Apple Silicon GPU (Metal) for acceleration
✅ OpenAI API key detected - using GPT-4o mini
⚠️ Note: OpenAI API responses may vary slightly even with the same inputs.
   For 100% reproducibility, save the model after generating labels.

🔄 Generating distinctive topic labels with LLM... This may take a minute.


100%|██████████| 21/21 [00:57<00:00,  2.75s/it]


✅ Topic labels generated successfully!

Topics with LLM-Generated Distinctive Labels:
    Topic  Count                                             Name
0      -1     31              -1_"Family-Friendly Fishing Access"
1       0     19            0_"Boat Launch Access and Facilities"
2       1     19           1_"Native Trout Experiences in Oregon"
3       2     18      2_Access and Affordability of Fishing Sites
4       3     11                    3_"Retention for Consumption"
5       4     11             4_"Guides and Regulatory Challenges"
6       5     10          5_"Quality and Size of Fish Population"
7       6      9  6_"Catch and Release Fly Fishing Opportunities"
8       7      8       7_"Family-Focused Lake Fishing Experience"
9       8      8       8_"Social Recommendations and Familiarity"
10      9      7                      9_"Backyard Nature Retreat"
11     10      7          10_Weather Impact on Fishing Conditions
12     11      6                11_"Accessible Fishing 

In [8]:
# Build hierarchical topics to show relationships between topics
hierarchical_topics = topic_model.hierarchical_topics(responses)

# Visualize the topic hierarchy
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("\nHierarchical topic structure created!")
print(f"Hierarchy levels: {len(hierarchical_topics['Distance'].unique())}")

100%|██████████| 19/19 [00:54<00:00,  2.85s/it]



Hierarchical topic structure created!
Hierarchy levels: 19


In [9]:
# Create a detailed topic relevance report
print("=" * 80)
print("TOPIC HIERARCHY AND RELEVANCE ANALYSIS")
print("=" * 80)

# Get topic info sorted by count (relevance)
topic_summary = topic_model.get_topic_info()
topic_summary = topic_summary[topic_summary['Topic'] != -1].sort_values('Count', ascending=False)

print(f"\n📊 Total Survey Responses: {len(responses)}")
print(f"📌 Topics Identified: {len(topic_summary)}")
print(f"🔍 Outlier Responses: {sum(1 for t in topics if t == -1)}")

print("\n" + "=" * 80)
print("TOPICS RANKED BY RELEVANCE (Number of Responses)")
print("=" * 80)

for idx, row in topic_summary.iterrows():
    topic_id = row['Topic']
    count = row['Count']
    percentage = (count / len(responses)) * 100
    
    print(f"\n🏆 Rank #{idx+1} | Topic {topic_id} | {count} responses ({percentage:.1f}%)")
    print(f"   Label: {row['Name']}")
    
    # Get top representative documents
    topic_docs = [responses[i] for i, t in enumerate(topics) if t == topic_id][:3]
    print(f"   Example responses:")
    for i, doc in enumerate(topic_docs, 1):
        print(f"     {i}. {doc[:80]}{'...' if len(doc) > 80 else ''}")
    print("-" * 80)

TOPIC HIERARCHY AND RELEVANCE ANALYSIS

📊 Total Survey Responses: 202
📌 Topics Identified: 20
🔍 Outlier Responses: 31

TOPICS RANKED BY RELEVANCE (Number of Responses)

🏆 Rank #2 | Topic 0 | 19 responses (9.4%)
   Label: 0_"Boat Launch Access and Facilities"
   Example responses:
     1. Access Boat ramps available
     2. launch ease, boat to campsite from bank accsess, toilets
     3. weather, season of year, boat launch capability
--------------------------------------------------------------------------------

🏆 Rank #3 | Topic 1 | 19 responses (9.4%)
   Label: 1_"Native Trout Experiences in Oregon"
   Example responses:
     1. access to fish
     2. Catching rainbow trout that are pink meat and native due to cut throat/rainbow m...
     3. fish
--------------------------------------------------------------------------------

🏆 Rank #4 | Topic 2 | 18 responses (8.9%)
   Label: 2_Access and Affordability of Fishing Sites
   Example responses:
     1. affordability
     2. ease of u

In [10]:
# Visualize topic distribution
fig_bar = topic_model.visualize_barchart(top_n_topics=10, height=400)
fig_bar.show()

print("\nTop 10 topics by document count visualized above")


Top 10 topics by document count visualized above


In [11]:
# Create an intertopic distance map to see relationships between topics
fig_map = topic_model.visualize_topics()
fig_map.show()

print("\nIntertopic distance map: Topics closer together are more semantically similar")


Intertopic distance map: Topics closer together are more semantically similar


In [12]:
# Export hierarchical structure to a DataFrame for further analysis
hierarchy_df = hierarchical_topics.copy()
hierarchy_df = hierarchy_df.sort_values('Distance')

print("Hierarchical Merging Structure (shows how topics combine):")
print("\nThis shows which topics merge together as you move up the hierarchy:")
print(hierarchy_df[['Parent_ID', 'Topics', 'Distance']].head(20).to_string())

print("\n\nNote: Lower distances indicate topics that are more similar and merge first")
print("As distance increases, broader topic categories are formed")

Hierarchical Merging Structure (shows how topics combine):

This shows which topics merge together as you move up the hierarchy:
   Parent_ID                                                                  Topics  Distance
0         20                                                                 [2, 11]  0.718759
1         21                                                                  [1, 3]  0.832341
2         22                                                               [1, 3, 5]  0.877655
3         23                                                                [12, 15]  0.883180
4         24                                                                 [0, 10]  0.910822
5         25                                                                 [9, 13]  0.928504
6         26                                                                 [6, 16]  0.937496
7         27                                                                 [4, 14]  0.943093
8         28    